# Pandas Applied — Merge

이번 질문은 현재 파일 하나만으로 답할 수 없습니다.

> **VIP 고객은 정말 더 많이 구매합니까?**

필요한 정보가 어디에 있는지 찾고, 공통 연결고리로 데이터를 합친 뒤, 연결 결과를 검증합니다.

## 독립 실행 준비

이 Notebook은 앞 Notebook의 실행 상태를 사용하지 않습니다. 아래 셀은 같은 기준으로 분석용 주문을 다시 구성합니다.

In [1]:
# colab 사용하시는 분들만!!!!
# 아래 코드 주석 해제해서 돌리셈

# import os
# os.makedirs('data', exist_ok=True)
#
# from google.colab import files
# uploaded = files.upload()   # 파일 선택창이 뜹니다
#
# # 업로드된 파일을 data/ 로 옮기기
# for name in uploaded:
#     os.rename(name, f'data/{name}')
#
# print(os.listdir('data'))

In [2]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
customers = pd.read_csv("data/customers.csv")

orders["sales"] = orders["price"] * orders["quantity"]
orders["discount"] = orders["discount"].fillna(0)
orders["final_sales"] = orders["sales"] * (1 - orders["discount"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["region"] = orders["region"].str.strip()

logical_orders = (
    orders.loc[orders["record_status"].ne("ingestion_error")]
    .drop_duplicates(subset="order_id", keep="first")
    .copy()
)

pd.Series({
    "logical_rows": len(logical_orders),
    "unique_order_id": logical_orders["order_id"].nunique(),
})

logical_rows       992
unique_order_id    992
dtype: int64

# Mission 1 — 현재 주문 데이터의 한계

`logical_orders`에 어떤 열이 있는지 확인하고, membership 정보가 있는지 찾아보세요.

In [3]:
# logical_orders의 열 이름을 확인하세요.
pass

In [4]:
logical_orders.columns

Index(['order_id', 'customer_id', 'order_date', 'product_id', 'product_name',
       'category', 'price', 'quantity', 'discount', 'payment_method', 'region',
       'channel', 'record_status', 'sales', 'final_sales'],
      dtype='str')

주문에는 고객을 가리키는 `customer_id`가 있지만 membership은 없습니다. 분석에 필요한 정보가 항상 한 파일에 모두 있는 것은 아닙니다.

# Mission 2 — 고객 파일 살펴보기

바로 연결하지 말고 `customers` 자체를 먼저 확인하세요.

In [5]:
# customers의 처음 몇 행을 확인하세요.
pass

In [6]:
# customers의 크기와 열 이름을 확인하세요.
pass

In [7]:
customers.head()

,customer_id,customer_name,age_group,home_region,signup_date,membership,marketing_opt_in
0,C0001,고객 001,40대,대구,2025-01-01,BASIC,True
1,C0002,고객 002,40대,서울,2025-02-02,BASIC,True
2,C0003,고객 003,40대,서울,2026-03-03,BASIC,True
3,C0004,고객 004,20대,부산,2025-04-04,BASIC,True
4,C0005,고객 005,30대,경기,2025-05-05,BASIC,True


In [8]:
customers.shape

(300, 7)

In [9]:
customers.columns

Index(['customer_id', 'customer_name', 'age_group', 'home_region',
       'signup_date', 'membership', 'marketing_opt_in'],
      dtype='str')

### Customer Evidence

- 고객 수: ______
- 고객 ID 열: __________________
- 회원 등급 열: __________________

## 공통 연결고리 찾기

두 데이터에 공통으로 존재하면서 같은 고객을 가리키는 열은 무엇인가요?

____________________

### Key

두 테이블을 연결하려면 같은 대상을 가리키는 공통 정보가 필요합니다. 이번 데이터에서는 `customer_id`가 연결고리입니다.

# Mission 3 — 작은 데이터로 Merge 예상하기

### orders_small

| customer_id | sales |
|---|---:|
| C01 | 50000 |
| C02 | 30000 |
| C03 | 70000 |

### customers_small

| customer_id | membership |
|---|---|
| C01 | VIP |
| C02 | BASIC |
| C03 | PLUS |

두 표를 `customer_id`로 연결하면 아래 표가 어떻게 채워질지 먼저 예상하세요.

| customer_id | sales | membership |
|---|---:|---|
| C01 | | |
| C02 | | |
| C03 | | |

In [10]:
orders_small = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "sales": [50000, 30000, 70000],
})

customers_small = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "membership": ["VIP", "BASIC", "PLUS"],
})

In [23]:
orders_small

,customer_id,sales
0,C01,50000
1,C02,30000
2,C03,70000


In [24]:
customers_small

,customer_id,membership
0,C01,VIP
1,C02,BASIC
2,C03,PLUS


In [11]:
orders_small.merge(
    customers_small,
    on="customer_id",
    how="left"
)

,customer_id,sales,membership
0,C01,50000,VIP
1,C02,30000,BASIC
2,C03,70000,PLUS


## Merge Mental Model

`주문 데이터 + 고객 데이터 + 공통 Key → 하나의 분석 가능한 데이터`

`how="left"`는 이번 분석의 기준인 주문을 유지하면서 찾을 수 있는 고객 정보를 옆에 붙입니다.

# Mission 4 — 실제 데이터 연결

현재 분석용 주문은 992행입니다. 고객 정보를 left merge하면 행 수가 줄어들지, 유지될지, 늘어날지 먼저 예상하고 이유를 적으세요.

In [12]:
# logical_orders와 customers를 customer_id 기준으로 left merge하고 merged에 저장하세요.
pass

<details>
<summary>힌트 보기</summary>

`left_dataframe.merge(right_dataframe, on="공통열", how="left")` 형태를 사용합니다.

</details>

In [13]:
merged = logical_orders.merge(
    customers,
    on="customer_id",
    how="left"
)

merged.head()

,order_id,customer_id,order_date,product_id,product_name,category,price,quantity,discount,payment_method,...,channel,record_status,sales,final_sales,customer_name,age_group,home_region,signup_date,membership,marketing_opt_in
0,ORD000001,C0291,2026-07-16,A004,토트백,패션,32000,5,0.00,카드,...,Web,verified,160000,160000.0,고객 291,40대,부산,2025-03-11,VIP,False
1,ORD000002,C0254,2026-07-31,A003,캡,패션,25000,1,0.00,카드,...,Web,verified,25000,25000.0,고객 254,30대,서울,2025-02-02,PLUS,True
2,ORD000003,C0223,2026-07-05,S004,플래너,문구,12000,7,0.05,기타,...,App,verified,84000,79800.0,고객 223,20대,서울,2024-07-27,PLUS,True
3,ORD000004,C0074,2026-07-12,E002,마우스,전자기기,29000,3,0.05,카카오페이,...,App,verified,87000,82650.0,고객 074,50대 이상,부산,2025-02-18,BASIC,True
4,ORD000005,C0049,2026-07-02,E005,USB 허브,전자기기,39000,5,0.05,네이버페이,...,Web,verified,195000,185250.0,고객 049,40대,서울,2025-01-21,BASIC,True


In [14]:
# merged의 행과 열 개수를 확인하세요.
pass

In [15]:
merged.shape

(992, 21)

### Merge 직후 확인

- Merge 전 주문 수: ______
- Merge 후 행 수: ______
- 행 수는 줄었다 / 같다 / 늘었다 중 무엇인가요?

# Mission 5 — 실행됐다고 성공일까요?

코드가 오류 없이 실행되었더라도 모든 고객 정보가 연결되었다고 바로 말할 수는 없습니다.

In [16]:
# merge 후 membership이 비어 있는 주문이 있는지 확인하세요.
pass

In [17]:
merged["membership"].isna().sum()

np.int64(10)

In [18]:
# membership이 비어 있는 주문의 customer_id를 확인하세요.
pass

왜 일부 membership이 비어 있을까요? 가능한 데이터 원인을 적어보세요.

________________________________________________

In [19]:
merge_check = logical_orders.merge(
    customers,
    on="customer_id",
    how="left",
    indicator=True
)

merge_check["_merge"].value_counts()

_merge
both          982
left_only      10
right_only      0
Name: count, dtype: int64

`both`는 양쪽 파일에서 고객을 찾은 주문이고, `left_only`는 주문에는 있지만 고객 파일에서는 찾지 못한 주문입니다.

## Merge Evidence

- 전체 주문: ______
- 고객 정보 연결 성공: ______
- 고객 정보 연결 실패: ______

`코드 실행 성공`과 `데이터 연결 성공`은 같은 말인가요? __________________

# Mission 6 — Membership 분석

'더 많이 구매한다'는 문장을 다음 세 질문으로 나눕니다.

1. 주문 건수가 많은가?
2. 총매출 기여가 큰가?
3. 주문 한 번의 평균 금액이 큰가?

In [20]:
# membership별 final_sales의 count, sum, mean을 확인하세요.
pass

<details>
<summary>힌트 보기</summary>

카테고리 분석에서 사용한 `groupby + agg` 구조를 다시 사용할 수 있습니다.

</details>

In [21]:
membership_summary = (
    merged
    .groupby("membership")["final_sales"]
    .agg(["count", "sum", "mean"])
)
membership_summary

,count,sum,mean
membership,,,
BASIC,500,25500000.0,51000.0
PLUS,332,27224000.0,82000.0
VIP,150,19926000.0,132840.0


### Membership Evidence

- 주문 건수 1위: ____________________
- 총매출 1위: ____________________
- 평균 주문금액 1위: ____________________

다음 문장을 데이터가 보여주는 범위에 맞게 더 정확하게 바꿔보세요.

> VIP가 더 많이 구매합니다.

________________________________________________

# Applied Final Mission

> **고객 등급별 구매 행동은 어떻게 다릅니까?**

주문 건수, 총매출, 평균 주문금액, 고객 정보를 찾지 못한 주문 수를 근거로 사용하세요.

In [22]:
# 필요한 분석 코드를 작성하세요.
pass

# 고객 등급 분석 결과

## Evidence 1 — 주문 건수
Claim: ____________________  
Number: ____________________

## Evidence 2 — 총매출
Claim: ____________________  
Number: ____________________

## Evidence 3 — 평균 주문금액
Claim: ____________________  
Number: ____________________

## Data Quality Note
고객 정보를 연결하지 못한 주문: ______건

## 팀장에게 한 문장으로 말한다면?

____________________________________________________________

# Merge에서 배운 흐름

`필요한 정보가 없음 → 다른 데이터 확인 → 공통 Key 찾기 → Merge → 행 수 확인 → 연결 실패 확인 → 분석 → Evidence`

> **Merge는 실행한 뒤 반드시 검증합니다.**

# 다음 단계

처음 보는 `unseen.csv`와 `unseen_customers.csv`에서 어떤 Pandas를 사용할지도 직접 결정합니다.